# 1. Kiểm tra GPU
Notebook này nên chạy trên Google Colab T4/A100.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free/1e9:.2f} GB / total: {total/1e9:.2f} GB')
else:
    print('NO GPU - chỉ nên dùng để setup hoặc test rất ngắn')


# 2. Mount Google Drive tùy chọn
Dùng nếu muốn cache model qua nhiều session.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'


# 3. Clone repo và cài dependencies


In [ ]:
%cd /content
!rm -rf Fast-VietTTS
!git clone https://github.com/baobui0709/Fast-VietTTS.git
%cd Fast-VietTTS

!pip install -q --upgrade pip
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchaudio==2.6.0
!pip install -q numpy==1.26.4 librosa soundfile num2words tqdm huggingface_hub gradio
!pip install -q s3tokenizer omegaconf conformer safetensors
!pip uninstall -y -q torchvision torchtext || true

%cd /content
!rm -rf TienHiep-TTS
!git clone https://github.com/yingchunbin/TienHiep-TTS.git
%cd /content/Fast-VietTTS


# 4. Import helper chung


In [ ]:
import sys, os, time, torch, torchaudio
from IPython.display import Audio, display

viterbox_src = '/content/TienHiep-TTS'
if viterbox_src not in sys.path:
    sys.path.insert(0, viterbox_src)

from src.engine import FastVietTTS
from src.audio_utils import prepare_reference, get_audio_duration


# 5. Load Fast-VietTTS fallback


In [ ]:
tts = FastVietTTS(device='cuda' if torch.cuda.is_available() else 'cpu', use_fp16=False, compile_model=False)
print('Model loaded')


# 6. Benchmark 3 độ dài text


In [ ]:
TESTS = {
    'short_50': 'Xin chào, đây là bài kiểm tra ngắn cho Fast-VietTTS.',
    'medium_200': 'Tiêu Viêm đứng trước cổng học viện, ánh mắt nhìn về phía xa, lòng tràn đầy quyết tâm bước vào con đường tu luyện mới. Hắn biết rằng phía trước còn rất nhiều thử thách đang chờ đợi.',
    'long_500': 'Việt Nam là một quốc gia nằm ở Đông Nam Á với lịch sử lâu đời, văn hóa phong phú và con người thân thiện. Từ những cánh đồng lúa trải dài ở đồng bằng sông Cửu Long đến những dãy núi hùng vĩ phía Bắc, mỗi vùng đất đều có câu chuyện riêng. Công nghệ trí tuệ nhân tạo đang mở ra nhiều cơ hội mới cho việc bảo tồn ngôn ngữ, tạo sách nói và hỗ trợ người dùng tiếp cận tri thức dễ dàng hơn qua giọng nói tự nhiên.',
}
results = []
for name, text in TESTS.items():
    t0 = time.time()
    wav = tts.generate(text, language='vi')
    elapsed = time.time() - t0
    dur = wav.shape[-1] / 24000
    results.append({'case': name, 'chars': len(text), 'audio_sec': dur, 'elapsed_sec': elapsed, 'rtf': elapsed/max(dur,1e-6)})
results


# 7. Benchmark có/không reference audio


In [ ]:
# Upload reference nếu muốn test voice clone
from google.colab import files
uploaded = files.upload()
ref = None
if uploaded:
    ref = prepare_reference(next(iter(uploaded.keys())), output_path='/content/ref_bench.wav')
    text = TESTS['short_50']
    t0 = time.time()
    wav = tts.generate(text, audio_prompt=ref, language='vi')
    elapsed = time.time() - t0
    dur = wav.shape[-1] / 24000
    results.append({'case': 'short_with_ref', 'chars': len(text), 'audio_sec': dur, 'elapsed_sec': elapsed, 'rtf': elapsed/max(dur,1e-6)})


# 8. Biểu đồ và bảng kết quả


In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.DataFrame(results)
display(df)
plt.figure(figsize=(8,4))
plt.bar(df['case'], df['rtf'])
plt.ylabel('RTF')
plt.title('Benchmark RTF Fast-VietTTS')
plt.xticks(rotation=30, ha='right')
plt.show()


# 9. Lưu kết quả vào docs/BENCHMARK.md


In [ ]:
from pathlib import Path
md = '# Benchmark — Fast-VietTTS

'
md += '## Results

'
md += df.to_markdown(index=False)
md += '

## Notes

- Benchmark chạy trong Colab runtime hiện tại.
- Model strategy: ViterBox fallback.
'
Path('docs/BENCHMARK.md').write_text(md, encoding='utf-8')
print(Path('docs/BENCHMARK.md').read_text(encoding='utf-8'))
print('Tóm tắt: Benchmark hoàn tất')
